# Tools in MCP
Companion notebook for the **Tools in MCP** section of Chapter 5 (Tool descriptors, Message format, Tool invocation flow).

MCP tools are not just Python functions -- they are *described* capabilities: a name, a description, and a JSON Schema for their input (and optionally output). This notebook builds and validates the exact descriptor and JSON-RPC messages shown in the chapter, using nothing but the standard library -- no server needed.

## Tool descriptor
A JSON object the server hands the client so it knows how to call the tool.

In [1]:
import json

tool_descriptor = {
    "name": "search",
    "description": "Search the web for information",
    "inputSchema": {
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "limit": {"type": "integer"},
        },
        "required": ["query"],
    },
}

print(json.dumps(tool_descriptor, indent=2))

{
  "name": "search",
  "description": "Search the web for information",
  "inputSchema": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string"
      },
      "limit": {
        "type": "integer"
      }
    },
    "required": [
      "query"
    ]
  }
}


A tiny validator (not a full JSON Schema engine, just enough to show the idea): checks required fields are present and that basic types (`string` / `integer`) match.

In [2]:
_PY_TYPE = {"string": str, "integer": int, "number": (int, float), "boolean": bool}


def validate_call(schema: dict, arguments: dict) -> list[str]:
    """Returns a list of validation errors (empty list = valid)."""
    errors = []
    for field in schema.get("required", []):
        if field not in arguments:
            errors.append(f"missing required field: {field}")

    for field, value in arguments.items():
        expected = schema["properties"].get(field, {}).get("type")
        py_type = _PY_TYPE.get(expected)
        if py_type and not isinstance(value, py_type):
            errors.append(f"field '{field}' should be {expected}, got {type(value).__name__}")

    return errors


schema = tool_descriptor["inputSchema"]

valid_call = {"query": "latest education policy updates", "limit": 5}
invalid_call = {"limit": "five"}  # missing "query", and "limit" is a string not an integer

print("valid_call errors:  ", validate_call(schema, valid_call))
print("invalid_call errors:", validate_call(schema, invalid_call))

valid_call errors:   []
invalid_call errors: ['missing required field: query', "field 'limit' should be integer, got str"]


## Message format (JSON-RPC 2.0)
Every MCP message carries a method name, parameters, and an id. Here is the `tools/call` request for the `search` tool above:

In [3]:
request = {
    "jsonrpc": "2.0",
    "method": "tools/call",
    "params": {
        "name": "search",
        "arguments": {
            "query": "latest education policy updates",
            "limit": 5,
        },
    },
    "id": 1,
}

print(json.dumps(request, indent=2))

{
  "jsonrpc": "2.0",
  "method": "tools/call",
  "params": {
    "name": "search",
    "arguments": {
      "query": "latest education policy updates",
      "limit": 5
    }
  },
  "id": 1
}


A successful response echoes the same `id` and carries the tool's output in `result`:

In [4]:
success_response = {
    "jsonrpc": "2.0",
    "result": {
        "items": ["Policy document A", "Policy document B"],
    },
    "id": 1,
}

print(json.dumps(success_response, indent=2))

{
  "jsonrpc": "2.0",
  "result": {
    "items": [
      "Policy document A",
      "Policy document B"
    ]
  },
  "id": 1
}


If something goes wrong (bad input, permission denied, a network failure), the server returns a structured `error` instead of a `result` -- never both:

In [5]:
error_response = {
    "jsonrpc": "2.0",
    "error": {
        "code": -32602,  # JSON-RPC "Invalid params"
        "message": "Invalid params: 'query' is required",
    },
    "id": 1,
}

print(json.dumps(error_response, indent=2))

assert ("result" in success_response) != ("result" in error_response or "error" in success_response)
print("\nA response carries exactly one of result / error -- never both, matching JSON-RPC 2.0.")

{
  "jsonrpc": "2.0",
  "error": {
    "code": -32602,
    "message": "Invalid params: 'query' is required"
  },
  "id": 1
}

A response carries exactly one of result / error -- never both, matching JSON-RPC 2.0.
